# ENHANCED EDA - Phase 1: Critical Missing Charts
## Objective:
#### "To analyze how student demographics, academic performance, and learning behaviors influence course completion and student success, in order to identify at-risk student profiles and recommend targeted academic support interventions."

---

## 📊 Phase 1 Additions:
1. **Completion & Retention Analysis** (3-4 charts)
2. **Time Series Progression** (3-4 charts)
3. **Multi-dimensional Comparisons** (2-3 charts)
4. **Course Deep Dives** (2-3 charts)

In [3]:
# ==========================================
# SETUP & DATA LOADING
# ==========================================
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# Set dark theme
pio.templates.default = "plotly_dark"

# Load data
master_df = pd.read_csv('../cleaned_data/master_dataset.csv')

# Data preparation
master_df['COMPLETION DATE'] = pd.to_datetime(master_df['COMPLETION DATE'], errors='coerce')
master_df['COMMENCEMENT DATE'] = pd.to_datetime(master_df['COMMENCEMENT DATE'], errors='coerce')

# Create helper columns
master_df['Completed'] = master_df['COMPLETION DATE'].notna().astype(int)
master_df['Pass_Status'] = master_df['GPA'].apply(lambda x: 'Pass' if pd.notna(x) and x >= 2.5 else ('Fail' if pd.notna(x) else 'No Grade'))
master_df['Risk_Status'] = master_df['GPA'].apply(lambda x: 'High Risk' if pd.notna(x) and x < 2.5 else ('At Risk' if pd.notna(x) and x < 3.0 else ('Safe' if pd.notna(x) else 'No Grade')))
master_df['Age_Group'] = pd.cut(master_df['AGE'], bins=[0, 25, 35, 45, 55, 100], labels=['18-25', '26-35', '36-45', '46-55', '56+'])
master_df['Attendance_Category'] = pd.cut(master_df['ATTENDANCE'], bins=[0, 70, 85, 100], labels=['Low (<70%)', 'Medium (70-85%)', 'High (85%+)'])

# Standardize funding source
master_df['COURSE FUNDING'] = master_df['COURSE FUNDING'].replace({
    'Individual - SFC': 'Individual-SFC',
    'Indivodual': 'Individual',
    'Indvidual - SFC': 'Individual-SFC',
    'Sponsored - SDF': 'Sponsored',
    'Sponsored-no SDF': 'Sponsored - no SDF'
})
master_df['Funding_Type'] = master_df['COURSE FUNDING'].apply(lambda x: 'Sponsored' if 'Sponsored' in str(x) else 'Individual')

print(f"Data loaded: {len(master_df)} records, {master_df['STUDENT ID'].nunique()} unique students")
print(f"Courses: {master_df['CLASS'].nunique()} unique courses")

Data loaded: 520 records, 295 unique students
Courses: 30 unique courses


---
# 📉 SECTION 1: COMPLETION & RETENTION ANALYSIS
**Critical Gap:** Understanding dropout patterns and completion rates

In [4]:
# ==========================================
# CHART 1.1: Completion Rate by Demographics
# ==========================================
# Calculate completion rate per student
student_completion = master_df.groupby('STUDENT ID').agg({
    'Completed': 'max',
    'AGE': 'first',
    'Age_Group': 'first',
    'NATIONALITY_STATUS': 'first',
    'GENDER': 'first',
    'Funding_Type': 'first'
}).reset_index()

# Completion by age group
completion_age = student_completion.groupby('Age_Group')['Completed'].agg(['mean', 'count']).reset_index()
completion_age['Completion_Rate'] = completion_age['mean'] * 100
completion_age['Dropout_Rate'] = (1 - completion_age['mean']) * 100

fig = go.Figure()

fig.add_trace(go.Bar(
    name='Completed',
    x=completion_age['Age_Group'],
    y=completion_age['Completion_Rate'],
    marker_color='#2ecc71',
    text=completion_age['Completion_Rate'].round(1),
    texttemplate='%{text}%',
    textposition='inside'
))

fig.add_trace(go.Bar(
    name='Dropped Out',
    x=completion_age['Age_Group'],
    y=completion_age['Dropout_Rate'],
    marker_color='#e74c3c',
    text=completion_age['Dropout_Rate'].round(1),
    texttemplate='%{text}%',
    textposition='inside'
))

fig.update_layout(
    title='<b>1.1 Completion vs Dropout Rate by Age Group</b><br><i>(Who is more likely to drop out?)</i>',
    barmode='stack',
    yaxis_title='Percentage (%)',
    xaxis_title='Age Group',
    height=500,
    showlegend=True
)

fig.show()

C:\Users\Thomas\AppData\Local\Temp\ipykernel_14480\4190788077.py:15: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  completion_age = student_completion.groupby('Age_Group')['Completed'].agg(['mean', 'count']).reset_index()


In [5]:
# ==========================================
# CHART 1.2: Retention Funnel Analysis
# ==========================================
# Calculate students present in each semester
retention = master_df.groupby('PERIOD')['STUDENT ID'].nunique().reset_index()
retention.columns = ['Semester', 'Student_Count']

# Order properly
semester_order = ['Sem 1', 'Sem 2', 'Sem 3', 'Sem 4']
retention['Semester'] = pd.Categorical(retention['Semester'], categories=semester_order, ordered=True)
retention = retention.sort_values('Semester')

# Calculate retention rate
retention['Retention_Rate'] = (retention['Student_Count'] / retention['Student_Count'].iloc[0] * 100).round(1)
retention['Dropout_Count'] = retention['Student_Count'].iloc[0] - retention['Student_Count']

fig = go.Figure()

# Funnel showing retention
fig.add_trace(go.Funnel(
    y=retention['Semester'],
    x=retention['Student_Count'],
    textposition="inside",
    textinfo="value+percent initial",
    opacity=0.85,
    marker={"color": ["#2ecc71", "#f39c12", "#e67e22", "#e74c3c"]},
    connector={"line": {"color": "royalblue", "dash": "dot", "width": 3}}
))

fig.update_layout(
    title='<b>1.2 Student Retention Funnel: Semester Progression</b><br><i>(How many students remain enrolled each semester?)</i>',
    height=500
)

fig.show()

print("\n📊 Retention Analysis:")
print(retention[['Semester', 'Student_Count', 'Retention_Rate', 'Dropout_Count']])


📊 Retention Analysis:
  Semester  Student_Count  Retention_Rate  Dropout_Count
0    Sem 1            280           100.0              0
1    Sem 2            148            52.9            132
2    Sem 3             75            26.8            205
3    Sem 4              2             0.7            278


In [6]:
# ==========================================
# CHART 1.3: Dropout Timing Analysis
# ==========================================
# Identify when students drop (their last recorded semester)
last_sem = master_df.groupby('STUDENT ID').agg({
    'PERIOD': 'last',
    'Completed': 'max',
    'Age_Group': 'first'
}).reset_index()

# Students who didn't complete
dropouts = last_sem[last_sem['Completed'] == 0]
dropout_timing = dropouts.groupby(['PERIOD', 'Age_Group']).size().reset_index(name='Count')

fig = px.bar(
    dropout_timing,
    x='PERIOD',
    y='Count',
    color='Age_Group',
    title='<b>1.3 Dropout Timing: When Do Students Leave?</b><br><i>(Last recorded semester before dropout, by age group)</i>',
    labels={'PERIOD': 'Last Semester Attended', 'Count': 'Number of Dropouts'},
    category_orders={'PERIOD': semester_order},
    color_discrete_sequence=px.colors.sequential.Reds_r,
    height=500
)

fig.update_layout(barmode='stack')
fig.show()

print(f"\n📊 Total Dropouts: {len(dropouts)} students")
print(f"Most critical dropout point: {dropout_timing.groupby('PERIOD')['Count'].sum().idxmax()}")

C:\Users\Thomas\AppData\Local\Temp\ipykernel_14480\4261996171.py:13: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.




📊 Total Dropouts: 26 students
Most critical dropout point: Sem 3


In [7]:
# ==========================================
# CHART 1.4: Time-to-Completion Analysis
# ==========================================
# Calculate actual duration vs expected duration
completed_students = master_df[master_df['Completed'] == 1].groupby('STUDENT ID').agg({
    'COURSE_DURATION_DAYS': 'first',
    'Age_Group': 'first',
    'PERIOD': 'count'  # Number of semesters recorded
}).reset_index()
completed_students.columns = ['STUDENT ID', 'Expected_Duration_Days', 'Age_Group', 'Semesters_Taken']

# Expected semesters (assuming ~170 days per semester)
completed_students['Expected_Semesters'] = (completed_students['Expected_Duration_Days'] / 170).round(0)
completed_students['On_Time'] = completed_students['Semesters_Taken'] <= completed_students['Expected_Semesters']

# Distribution
ontime_dist = completed_students.groupby('Age_Group')['On_Time'].value_counts(normalize=True).unstack().fillna(0) * 100
ontime_dist = ontime_dist.reset_index()

fig = go.Figure()

fig.add_trace(go.Bar(
    name='On-Time Completion',
    x=ontime_dist['Age_Group'],
    y=ontime_dist.get(True, 0),
    marker_color='#2ecc71',
    text=ontime_dist.get(True, 0).round(1),
    texttemplate='%{text}%'
))

fig.add_trace(go.Bar(
    name='Extended Duration',
    x=ontime_dist['Age_Group'],
    y=ontime_dist.get(False, 0),
    marker_color='#f39c12',
    text=ontime_dist.get(False, 0).round(1),
    texttemplate='%{text}%'
))

fig.update_layout(
    title='<b>1.4 Time-to-Completion: On-Time vs Extended Duration</b><br><i>(Are students completing within expected timeframe?)</i>',
    barmode='stack',
    yaxis_title='Percentage (%)',
    xaxis_title='Age Group',
    height=500
)

fig.show()

C:\Users\Thomas\AppData\Local\Temp\ipykernel_14480\1669185348.py:17: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



---
# 📈 SECTION 2: TIME SERIES PROGRESSION ANALYSIS
**Critical Gap:** Individual student progression tracking over semesters

In [8]:
# ==========================================
# CHART 2.1: GPA Trajectory by Risk Category
# ==========================================
# Categorize students based on Sem 1 performance
sem1_performance = master_df[master_df['PERIOD'] == 'Sem 1'][['STUDENT ID', 'GPA']].copy()
sem1_performance['Sem1_Category'] = sem1_performance['GPA'].apply(
    lambda x: 'High Risk (GPA<2.5)' if pd.notna(x) and x < 2.5 else 
             ('At Risk (2.5-3.0)' if pd.notna(x) and x < 3.0 else 
             ('Safe (GPA≥3.0)' if pd.notna(x) else 'No Grade'))
)

# Merge back
trajectory_df = master_df.merge(sem1_performance[['STUDENT ID', 'Sem1_Category']], on='STUDENT ID', how='left')

# Calculate average GPA per semester per category
trajectory_avg = trajectory_df.groupby(['PERIOD', 'Sem1_Category'])['GPA'].mean().reset_index()
trajectory_avg['PERIOD'] = pd.Categorical(trajectory_avg['PERIOD'], categories=semester_order, ordered=True)
trajectory_avg = trajectory_avg.sort_values('PERIOD')

fig = px.line(
    trajectory_avg,
    x='PERIOD',
    y='GPA',
    color='Sem1_Category',
    markers=True,
    title='<b>2.1 GPA Trajectory: Do Students Improve Over Time?</b><br><i>(Tracking progression by initial performance level)</i>',
    labels={'PERIOD': 'Semester', 'GPA': 'Average GPA'},
    color_discrete_map={
        'High Risk (GPA<2.5)': '#e74c3c',
        'At Risk (2.5-3.0)': '#f39c12',
        'Safe (GPA≥3.0)': '#2ecc71'
    },
    height=500
)

# Add passing threshold line
fig.add_hline(y=2.5, line_dash="dash", line_color="white", 
              annotation_text="Passing Threshold (2.5)", annotation_position="right")

fig.show()

In [9]:
# ==========================================
# CHART 2.2: Attendance Patterns Over Time
# ==========================================
# Average attendance by semester and risk category
attendance_progression = trajectory_df.groupby(['PERIOD', 'Sem1_Category'])['ATTENDANCE'].mean().reset_index()
attendance_progression['PERIOD'] = pd.Categorical(attendance_progression['PERIOD'], categories=semester_order, ordered=True)
attendance_progression = attendance_progression.sort_values('PERIOD')

fig = px.line(
    attendance_progression,
    x='PERIOD',
    y='ATTENDANCE',
    color='Sem1_Category',
    markers=True,
    title='<b>2.2 Attendance Trends: Do At-Risk Students Improve Discipline?</b><br><i>(Tracking attendance over semesters by risk level)</i>',
    labels={'PERIOD': 'Semester', 'ATTENDANCE': 'Average Attendance (%)'},
    color_discrete_map={
        'High Risk (GPA<2.5)': '#e74c3c',
        'At Risk (2.5-3.0)': '#f39c12',
        'Safe (GPA≥3.0)': '#2ecc71'
    },
    height=500
)

fig.add_hline(y=85, line_dash="dash", line_color="yellow", 
              annotation_text="Target: 85%", annotation_position="right")

fig.show()

In [10]:
# ==========================================
# CHART 2.3: Recovery Rate Analysis
# ==========================================
# Students who failed Sem 1 but recovered by Sem 3
sem1_data = master_df[master_df['PERIOD'] == 'Sem 1'][['STUDENT ID', 'GPA']].copy()
sem1_data.columns = ['STUDENT ID', 'Sem1_GPA']

sem3_data = master_df[master_df['PERIOD'] == 'Sem 3'][['STUDENT ID', 'GPA']].copy()
sem3_data.columns = ['STUDENT ID', 'Sem3_GPA']

recovery_df = sem1_data.merge(sem3_data, on='STUDENT ID', how='inner')
recovery_df['Sem1_Failed'] = recovery_df['Sem1_GPA'] < 2.5
recovery_df['Sem3_Passed'] = recovery_df['Sem3_GPA'] >= 2.5
recovery_df['Recovered'] = recovery_df['Sem1_Failed'] & recovery_df['Sem3_Passed']

# Calculate recovery stats
failed_sem1 = recovery_df[recovery_df['Sem1_Failed']]
recovery_rate = (failed_sem1['Recovered'].sum() / len(failed_sem1) * 100) if len(failed_sem1) > 0 else 0

# Create recovery categories
recovery_df['Recovery_Status'] = recovery_df.apply(
    lambda x: 'Recovered (Failed→Passed)' if x['Recovered'] else
             ('Still Struggling (Failed→Failed)' if x['Sem1_Failed'] and not x['Sem3_Passed'] else
             ('Maintained Excellence (Passed→Passed)' if not x['Sem1_Failed'] and x['Sem3_Passed'] else
             'Declined (Passed→Failed)')), axis=1
)

recovery_counts = recovery_df['Recovery_Status'].value_counts()

fig = go.Figure(data=[
    go.Pie(
        labels=recovery_counts.index,
        values=recovery_counts.values,
        marker=dict(colors=['#2ecc71', '#e74c3c', '#3498db', '#f39c12']),
        textinfo='label+percent+value',
        hole=0.4
    )
])

fig.update_layout(
    title=f'<b>2.3 Recovery Rate Analysis: Sem 1 → Sem 3</b><br><i>(Recovery Rate for Failed Students: {recovery_rate:.1f}%)</i>',
    height=500
)

fig.show()

print(f"\n📊 Recovery Analysis:")
print(f"Students who failed Sem 1: {len(failed_sem1)}")
print(f"Students who recovered by Sem 3: {failed_sem1['Recovered'].sum()}")
print(f"Recovery rate: {recovery_rate:.1f}%")


📊 Recovery Analysis:
Students who failed Sem 1: 22
Students who recovered by Sem 3: 12
Recovery rate: 54.5%


In [11]:
# ==========================================
# CHART 2.4: Study Hours Evolution
# ==========================================
# Track how study hours change over semesters
study_evolution = trajectory_df.groupby(['PERIOD', 'Sem1_Category'])['SELF-STUDY HRS'].mean().reset_index()
study_evolution['PERIOD'] = pd.Categorical(study_evolution['PERIOD'], categories=semester_order, ordered=True)
study_evolution = study_evolution.sort_values('PERIOD')

fig = px.bar(
    study_evolution,
    x='PERIOD',
    y='SELF-STUDY HRS',
    color='Sem1_Category',
    barmode='group',
    title='<b>2.4 Study Hours Evolution: Do Students Increase Effort Over Time?</b><br><i>(Average self-study hours by semester and initial performance)</i>',
    labels={'PERIOD': 'Semester', 'SELF-STUDY HRS': 'Average Weekly Study Hours'},
    color_discrete_map={
        'High Risk (GPA<2.5)': '#e74c3c',
        'At Risk (2.5-3.0)': '#f39c12',
        'Safe (GPA≥3.0)': '#2ecc71'
    },
    height=500
)

fig.show()

---
# 🔍 SECTION 3: MULTI-DIMENSIONAL COMPARISONS
**Critical Gap:** Gender, funding, and qualification impact analysis

In [12]:
# ==========================================
# CHART 3.1: Gender × Age × Performance Analysis
# ==========================================
gender_age_perf = master_df.groupby(['GENDER', 'Age_Group'])['GPA'].mean().reset_index()

fig = px.bar(
    gender_age_perf,
    x='Age_Group',
    y='GPA',
    color='GENDER',
    barmode='group',
    title='<b>3.1 Gender Performance Gap: Does Age Affect Men and Women Differently?</b><br><i>(Average GPA by gender and age group)</i>',
    labels={'Age_Group': 'Age Group', 'GPA': 'Average GPA'},
    color_discrete_map={'F': '#e74c3c', 'M': '#3498db'},
    height=500
)

fig.add_hline(y=2.5, line_dash="dash", line_color="white", 
              annotation_text="Passing Threshold", annotation_position="right")

fig.show()

# Additional insights
print("\n📊 Gender Distribution:")
print(master_df['GENDER'].value_counts())
print("\nAverage GPA by Gender:")
print(master_df.groupby('GENDER')['GPA'].mean())

C:\Users\Thomas\AppData\Local\Temp\ipykernel_14480\1486626138.py:4: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.




📊 Gender Distribution:
GENDER
F    456
M     64
Name: count, dtype: int64

Average GPA by Gender:
GENDER
F    3.125792
M    3.057143
Name: GPA, dtype: float64


In [13]:
# ==========================================
# CHART 3.2: Funding Source Impact Analysis
# ==========================================
funding_impact = master_df.groupby(['Funding_Type', 'Pass_Status']).size().reset_index(name='Count')
funding_totals = master_df.groupby('Funding_Type').size().reset_index(name='Total')
funding_impact = funding_impact.merge(funding_totals, on='Funding_Type')
funding_impact['Percentage'] = (funding_impact['Count'] / funding_impact['Total'] * 100).round(1)

fig = px.bar(
    funding_impact,
    x='Funding_Type',
    y='Percentage',
    color='Pass_Status',
    barmode='stack',
    title='<b>3.2 Funding Source Impact: Does Sponsorship Affect Success?</b><br><i>(Pass/Fail rates by funding type)</i>',
    labels={'Funding_Type': 'Funding Source', 'Percentage': 'Percentage (%)'},
    color_discrete_map={'Pass': '#2ecc71', 'Fail': '#e74c3c', 'No Grade': '#95a5a6'},
    height=500,
    text='Percentage'
)

fig.update_traces(texttemplate='%{text}%', textposition='inside')
fig.show()

# Company support comparison
print("\n📊 Company Support by Funding Type:")
print(master_df.groupby('Funding_Type')['COMPANY SUPPORT'].mean())


📊 Company Support by Funding Type:
Funding_Type
Individual    3.863291
Sponsored     3.878788
Name: COMPANY SUPPORT, dtype: float64


In [14]:
# ==========================================
# CHART 3.3: Qualification vs Actual Performance
# ==========================================
# Simplified qualification categories
master_df['Qualification_Level'] = master_df['HIGHEST QUALIFICATION'].apply(
    lambda x: 'Degree' if x == 'Degree' else ('Diploma' if x == 'Diploma' else 'Certificate')
)

qual_performance = master_df.groupby(['Qualification_Level', 'Risk_Status']).size().reset_index(name='Count')
qual_totals = master_df.groupby('Qualification_Level').size().reset_index(name='Total')
qual_performance = qual_performance.merge(qual_totals, on='Qualification_Level')
qual_performance['Percentage'] = (qual_performance['Count'] / qual_performance['Total'] * 100).round(1)

fig = px.bar(
    qual_performance,
    x='Qualification_Level',
    y='Percentage',
    color='Risk_Status',
    barmode='stack',
    title='<b>3.3 Qualification Paradox: Do Credentials Predict Success?</b><br><i>(Risk distribution by prior qualification level)</i>',
    labels={'Qualification_Level': 'Prior Qualification', 'Percentage': 'Percentage (%)'},
    color_discrete_map={'Safe': '#2ecc71', 'At Risk': '#f39c12', 'High Risk': '#e74c3c', 'No Grade': '#95a5a6'},
    category_orders={'Qualification_Level': ['Certificate', 'Diploma', 'Degree']},
    height=500,
    text='Percentage'
)

fig.update_traces(texttemplate='%{text}%', textposition='inside')
fig.show()

print("\n📊 Average GPA by Qualification:")
print(master_df.groupby('Qualification_Level')['GPA'].mean().sort_values(ascending=False))


📊 Average GPA by Qualification:
Qualification_Level
Diploma        3.200000
Degree         3.129353
Certificate    3.042286
Name: GPA, dtype: float64


---
# 🎯 SECTION 4: COURSE DEEP DIVES
**Critical Gap:** Individual course performance profiles and difficulty analysis

In [15]:
# ==========================================
# CHART 4.1: Course Performance Radar Chart
# ==========================================
# Calculate course-level metrics
course_metrics = master_df.groupby('CLASS').agg({
    'GPA': 'mean',
    'ATTENDANCE': 'mean',
    'PRIOR KNOWLEDGE': 'mean',
    'COURSE RELEVANCE': 'mean',
    'TEACHING SUPPORT': 'mean',
    'SELF-STUDY HRS': 'mean'
}).reset_index()

# Normalize to 0-100 scale for radar chart
course_metrics['GPA_norm'] = (course_metrics['GPA'] / 4.0 * 100).round(1)
course_metrics['Knowledge_norm'] = (course_metrics['PRIOR KNOWLEDGE'] / 5.0 * 100).round(1)
course_metrics['Relevance_norm'] = (course_metrics['COURSE RELEVANCE'] / 5.0 * 100).round(1)
course_metrics['Teaching_norm'] = (course_metrics['TEACHING SUPPORT'] / 5.0 * 100).round(1)
course_metrics['Study_norm'] = (course_metrics['SELF-STUDY HRS'] / course_metrics['SELF-STUDY HRS'].max() * 100).round(1)

# Select top 5 courses by enrollment for radar
top_courses = master_df['CLASS'].value_counts().head(5).index.tolist()
course_radar = course_metrics[course_metrics['CLASS'].isin(top_courses)]

# Create radar chart
fig = go.Figure()

categories = ['GPA', 'Attendance', 'Prior Knowledge', 'Course Relevance', 'Teaching Support']

for course in top_courses:
    course_data = course_radar[course_radar['CLASS'] == course].iloc[0]
    
    fig.add_trace(go.Scatterpolar(
        r=[
            course_data['GPA_norm'],
            course_data['ATTENDANCE'],
            course_data['Knowledge_norm'],
            course_data['Relevance_norm'],
            course_data['Teaching_norm']
        ],
        theta=categories,
        fill='toself',
        name=course
    ))

fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 100]
        )
    ),
    title='<b>4.1 Course Performance Profiles: Multi-Dimensional Comparison</b><br><i>(Top 5 courses by enrollment - 100 = Best performance)</i>',
    showlegend=True,
    height=600
)

fig.show()

In [16]:
# ==========================================
# CHART 4.2: Course Difficulty Matrix
# ==========================================
# Calculate failure rate and avg GPA per course
course_difficulty = master_df.groupby('CLASS').agg({
    'GPA': ['mean', 'count'],
    'STUDENT ID': 'nunique'
}).reset_index()
course_difficulty.columns = ['CLASS', 'Avg_GPA', 'Records', 'Total_Students']

# Calculate failure rate
failures = master_df[master_df['GPA'] < 2.5].groupby('CLASS').size().reset_index(name='Failures')
course_difficulty = course_difficulty.merge(failures, on='CLASS', how='left').fillna(0)
course_difficulty['Failure_Rate'] = (course_difficulty['Failures'] / course_difficulty['Records'] * 100).round(1)

# Bubble chart: X=Avg GPA, Y=Failure Rate, Size=Enrollment
fig = px.scatter(
    course_difficulty,
    x='Avg_GPA',
    y='Failure_Rate',
    size='Total_Students',
    color='Failure_Rate',
    hover_data=['CLASS', 'Total_Students'],
    title='<b>4.2 Course Difficulty Matrix: Which Courses Need Intervention?</b><br><i>(Bubble size = Number of students enrolled)</i>',
    labels={'Avg_GPA': 'Average GPA', 'Failure_Rate': 'Failure Rate (%)', 'Total_Students': 'Enrollment'},
    color_continuous_scale='Reds',
    height=600
)

# Add quadrant lines
fig.add_vline(x=3.0, line_dash="dash", line_color="gray", annotation_text="GPA 3.0")
fig.add_hline(y=20, line_dash="dash", line_color="gray", annotation_text="20% Failure")

# Add annotations for quadrants
fig.add_annotation(x=3.5, y=30, text="High Difficulty", showarrow=False, font=dict(color="red", size=14))
fig.add_annotation(x=3.5, y=10, text="Manageable", showarrow=False, font=dict(color="green", size=14))

fig.show()

print("\n📊 Most Challenging Courses (by failure rate):")
print(course_difficulty.nlargest(5, 'Failure_Rate')[['CLASS', 'Avg_GPA', 'Failure_Rate', 'Total_Students']])


📊 Most Challenging Courses (by failure rate):
       CLASS   Avg_GPA  Failure_Rate  Total_Students
4   1102-001  2.704000          40.0               8
20  2102-070  2.863636          36.4              11
6   1102-003  2.985714          33.3               7
0   1101-009  2.985294          29.4              11
22  5112-009  2.845455          27.3              11


In [17]:
# ==========================================
# CHART 4.3: Course-Specific At-Risk Patterns
# ==========================================
# Identify at-risk student profiles by course
top_5_courses = master_df['CLASS'].value_counts().head(5).index.tolist()
course_risk = master_df[master_df['CLASS'].isin(top_5_courses)].copy()

# Risk breakdown by course and age
risk_breakdown = course_risk.groupby(['CLASS', 'Age_Group', 'Risk_Status']).size().reset_index(name='Count')
course_totals = course_risk.groupby(['CLASS', 'Age_Group']).size().reset_index(name='Total')
risk_breakdown = risk_breakdown.merge(course_totals, on=['CLASS', 'Age_Group'])
risk_breakdown['Percentage'] = (risk_breakdown['Count'] / risk_breakdown['Total'] * 100).round(1)

# Focus on high-risk students
high_risk_only = risk_breakdown[risk_breakdown['Risk_Status'] == 'High Risk']

fig = px.bar(
    high_risk_only,
    x='CLASS',
    y='Percentage',
    color='Age_Group',
    title='<b>4.3 Course-Specific Risk Profiles: Which Age Groups Struggle in Each Course?</b><br><i>(Percentage of HIGH RISK students by course and age)</i>',
    labels={'CLASS': 'Course', 'Percentage': 'High Risk Students (%)'},
    barmode='group',
    height=500,
    color_discrete_sequence=px.colors.sequential.Reds
)

fig.show()

C:\Users\Thomas\AppData\Local\Temp\ipykernel_14480\932843576.py:9: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

C:\Users\Thomas\AppData\Local\Temp\ipykernel_14480\932843576.py:10: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



---
# 📊 SUMMARY: Phase 1 Complete

## Charts Added:

### **Completion & Retention (4 charts):**
1. Completion vs Dropout Rate by Age Group
2. Retention Funnel Analysis
3. Dropout Timing Analysis
4. Time-to-Completion Analysis

### **Time Series Progression (4 charts):**
5. GPA Trajectory by Risk Category
6. Attendance Patterns Over Time
7. Recovery Rate Analysis
8. Study Hours Evolution

### **Multi-Dimensional Comparisons (3 charts):**
9. Gender × Age × Performance Analysis
10. Funding Source Impact Analysis
11. Qualification vs Actual Performance

### **Course Deep Dives (3 charts):**
12. Course Performance Radar Chart
13. Course Difficulty Matrix
14. Course-Specific At-Risk Patterns

**Total New Charts:** 14 charts

---

## Next Steps:
1. Review all charts with teammate
2. Score each chart for dashboard suitability
3. Select best 8 charts (4 per student)
4. Move to Phase 2: Enhance charts with interactivity (Graph Objects)